# Selective Dynamical Decoupling

Quantum hardware suffers from **decoherence**: qubits lose their quantum state over time, even when no gates are acting on them. Dynamical Decoupling (DD) is a technique that inserts carefully timed pulse sequences during idle periods to suppress this decoherence.

Qiskit does **not** enable DD at any optimization level by default. qb-compiler adds it as a guaranteed improvement on top of Qiskit's transpilation.

This notebook covers:

1. What dynamical decoupling is and why it matters
2. Compiling with and without DD using `compile_enhanced()`
3. Comparing estimated fidelity with and without DD
4. When DD helps (long idle periods) vs when it hurts (dense circuits)
5. Inspecting `EnhancedCompileResult` fields
6. XX vs XY4 sequences
7. Different circuit types and DD impact
8. Depth impact of DD insertion

## 1. What is Dynamical Decoupling?

When a qubit is idle (no gate is acting on it), it decoheres — its quantum state leaks into the environment. The rate of this leakage is characterized by two timescales:

- **T1 (relaxation time):** how long until an excited qubit decays to the ground state
- **T2 (dephasing time):** how long until the phase information is lost

Dynamical Decoupling works by inserting pairs of fast pulses (like X-X or X-Y-X-Y) during idle periods. These pulses effectively "refocus" the qubit, canceling out the slow drift caused by environmental noise.

### DD Sequences

| Sequence | Pulses | Suppresses | Best for |
|----------|--------|------------|----------|
| **XX** | X, X | Dephasing (T2) | General use, low overhead |
| **XY4** | X, Y, X, Y | Dephasing + amplitude damping | Short T2 qubits, stronger noise |

The XY4 sequence provides better suppression but adds more gates. qb-compiler can auto-select based on calibration data.

## 2. Setup

In [ ]:
import math

from qiskit import QuantumCircuit

from qb_compiler import BACKEND_CONFIGS, QBCompiler

## 3. Compiling With and Without DD

The `compile_enhanced()` method on `QBCompiler` uses Qiskit for layout and routing, then adds DD on top. It returns an `EnhancedCompileResult` that contains both the base circuit and the DD-enhanced circuit.

In [ ]:
# Build a GHZ circuit with some idle time
# The sequential CX chain means early qubits are idle while later ones are entangled
ghz = QuantumCircuit(6, 6, name="GHZ-6")
ghz.h(0)
for i in range(5):
    ghz.cx(i, i + 1)
ghz.measure(range(6), range(6))

print(f"Circuit: {ghz.name}")
print(f"Qubits:  {ghz.num_qubits}")
print(f"Depth:   {ghz.depth()}")
print(f"Ops:     {ghz.count_ops()}")

In [ ]:
compiler = QBCompiler.from_backend("ibm_fez")

# compile_enhanced needs a Qiskit Target for scheduling
# The compiler builds one from calibration data automatically
target = compiler.qiskit_target

if target is None:
    print("No Qiskit target available. Ensure calibration data is loaded.")
else:
    # Compile with DD (default XX sequence)
    result_dd = compiler.compile_enhanced(
        ghz,
        target,
        n_seeds=10,
        dd_type="XX",
    )

    print(f"=== Enhanced Compilation Result ===")
    print(f"Best seed:                 {result_dd.best_seed}")
    print(f"2Q gates (base):           {result_dd.two_qubit_gate_count}")
    print(f"2Q gates (with DD):        {result_dd.two_qubit_gate_count_with_dd}")
    print(f"DD gates inserted:         {result_dd.dd_gates_inserted}")
    print(f"DD type:                   {result_dd.dd_type}")
    print(f"Estimated fidelity (base): {result_dd.estimated_fidelity:.4f}")
    print(f"Estimated fidelity (DD):   {result_dd.estimated_fidelity_with_dd:.4f}")
    print(f"Compilation time:          {result_dd.compilation_time_ms:.2f} ms")
    print(f"Physical qubits:           {result_dd.physical_qubits}")

## 4. Fidelity Improvement from DD

DD improves fidelity by suppressing decoherence during idle periods. The improvement is largest for circuits with many idle slots.

In [ ]:
if target is not None:
    fid_base = result_dd.estimated_fidelity
    fid_dd = result_dd.estimated_fidelity_with_dd

    if fid_base > 0:
        improvement_pct = (fid_dd - fid_base) / fid_base * 100
    else:
        improvement_pct = 0

    print(f"Fidelity without DD: {fid_base:.4f}")
    print(f"Fidelity with DD:    {fid_dd:.4f}")
    print(f"Improvement:         {improvement_pct:+.2f}%")
    print(f"DD gates added:      {result_dd.dd_gates_inserted}")

## 5. XX vs XY4 Sequences

The XX sequence (X-X) is simpler and adds fewer gates. The XY4 sequence (X-Y-X-Y) provides stronger suppression but doubles the gate overhead.

qb-compiler can auto-select based on calibration data: if the median T2 of the device is below 50 microseconds, it uses XY4 for stronger protection.

In [ ]:
if target is not None:
    # Compare XX vs XY4
    result_xx = compiler.compile_enhanced(ghz, target, n_seeds=10, dd_type="XX")
    result_xy4 = compiler.compile_enhanced(ghz, target, n_seeds=10, dd_type="XY4")

    print(f"{'Metric':30s}  {'XX':>12s}  {'XY4':>12s}")
    print("-" * 58)
    print(f"{'DD gates inserted':30s}  {result_xx.dd_gates_inserted:>12d}  {result_xy4.dd_gates_inserted:>12d}")
    print(f"{'2Q gates (with DD)':30s}  {result_xx.two_qubit_gate_count_with_dd:>12d}  {result_xy4.two_qubit_gate_count_with_dd:>12d}")
    print(f"{'Fidelity (base)':30s}  {result_xx.estimated_fidelity:>12.4f}  {result_xy4.estimated_fidelity:>12.4f}")
    print(f"{'Fidelity (with DD)':30s}  {result_xx.estimated_fidelity_with_dd:>12.4f}  {result_xy4.estimated_fidelity_with_dd:>12.4f}")
    print(f"{'Compilation time (ms)':30s}  {result_xx.compilation_time_ms:>12.2f}  {result_xy4.compilation_time_ms:>12.2f}")

XY4 inserts more gates (roughly 2x the DD gates of XX) because each idle slot gets a 4-pulse sequence instead of 2. This can actually *increase* depth. The trade-off is worth it when the dominant noise source is low-frequency dephasing, which XY4 suppresses more effectively.

## 6. When DD Helps: Circuits with Long Idle Periods

DD is most effective when qubits sit idle for significant durations. A GHZ circuit is a good example: qubit 0 is idle while qubits 1-5 are being entangled.

In [ ]:
if target is not None:
    # Build GHZ circuits of increasing size
    # More qubits = longer idle time for early qubits = more DD benefit
    print(f"{'N':>4s}  {'Fid (base)':>12s}  {'Fid (DD)':>12s}  {'Improvement':>12s}  {'DD Gates':>10s}")
    print("-" * 56)

    for n in [3, 5, 8, 10, 15, 20]:
        qc = QuantumCircuit(n, n, name=f"GHZ-{n}")
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
        qc.measure(range(n), range(n))

        r = compiler.compile_enhanced(qc, target, n_seeds=5, dd_type="XX")
        impr = r.estimated_fidelity_with_dd - r.estimated_fidelity
        print(
            f"{n:>4d}  {r.estimated_fidelity:>12.4f}  "
            f"{r.estimated_fidelity_with_dd:>12.4f}  "
            f"{impr:>+12.4f}  {r.dd_gates_inserted:>10d}"
        )

## 7. When DD Hurts: Dense Circuits

Circuits where every qubit is active at every time step have no idle periods, so DD has nothing to fill. In very dense circuits, attempting DD can even slightly degrade fidelity because the DD pulses themselves have non-zero error.

In [ ]:
if target is not None:
    # Build a dense circuit: all qubits active at every layer
    dense = QuantumCircuit(4, 4, name="Dense-4")
    for _ in range(5):
        # All qubits get single-qubit gates
        for q in range(4):
            dense.h(q)
        # Parallel CX pairs (no idle qubits)
        dense.cx(0, 1)
        dense.cx(2, 3)
        for q in range(4):
            dense.rz(0.5, q)
    dense.measure(range(4), range(4))

    r_dense = compiler.compile_enhanced(dense, target, n_seeds=5, dd_type="XX")

    print(f"Dense circuit:")
    print(f"  Fidelity (base):   {r_dense.estimated_fidelity:.4f}")
    print(f"  Fidelity (DD):     {r_dense.estimated_fidelity_with_dd:.4f}")
    print(f"  DD gates inserted: {r_dense.dd_gates_inserted}")

    if r_dense.dd_gates_inserted == 0:
        print("  No idle slots found — DD had nothing to insert.")
    else:
        diff = r_dense.estimated_fidelity_with_dd - r_dense.estimated_fidelity
        print(f"  Fidelity change:   {diff:+.4f}")

## 8. DD Impact Across Different Circuit Types

Different algorithmic families have different idle-time profiles. Let's compare DD impact across GHZ, QFT, QAOA, and VQE ansatz circuits.

In [ ]:
def make_qft(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"QFT-{n}")
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            qc.cp(math.pi / (2 ** (j - i)), i, j)
    qc.measure(range(n), range(n))
    return qc


def make_qaoa(n: int, p: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"QAOA-{n}-p{p}")
    for q in range(n):
        qc.h(q)
    for _ in range(p):
        for i in range(n):
            j = (i + 1) % n
            qc.cx(i, j)
            qc.rz(0.5, j)
            qc.cx(i, j)
        for i in range(n):
            qc.rx(0.7, i)
    qc.measure(range(n), range(n))
    return qc


def make_vqe(n: int, layers: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"VQE-{n}-L{layers}")
    for layer in range(layers):
        for q in range(n):
            qc.ry(0.5 + layer * 0.1, q)
        for q in range(0, n - 1, 2):
            qc.cx(q, q + 1)
        for q in range(1, n - 1, 2):
            qc.cx(q, q + 1)
    qc.measure(range(n), range(n))
    return qc


print("Circuit builders defined.")

In [ ]:
if target is not None:
    n = 6
    test_circuits = [
        (lambda: (qc := QuantumCircuit(n, n, name=f"GHZ-{n}"),
                  qc.h(0),
                  [qc.cx(i, i+1) for i in range(n-1)],
                  qc.measure(range(n), range(n)),
                  qc)[-1])(),
        make_qft(n),
        make_qaoa(n, p=2),
        make_vqe(n, layers=3),
    ]

    print(f"{'Circuit':15s}  {'Fid(base)':>10s}  {'Fid(DD)':>10s}  {'Delta':>8s}  {'DD Gates':>10s}  {'Verdict':>10s}")
    print("-" * 70)

    for circ in test_circuits:
        r = compiler.compile_enhanced(circ, target, n_seeds=5, dd_type="XX")
        delta = r.estimated_fidelity_with_dd - r.estimated_fidelity
        verdict = "helps" if delta > 0.001 else ("neutral" if delta > -0.001 else "hurts")
        print(
            f"{circ.name:15s}  {r.estimated_fidelity:>10.4f}  "
            f"{r.estimated_fidelity_with_dd:>10.4f}  {delta:>+8.4f}  "
            f"{r.dd_gates_inserted:>10d}  {verdict:>10s}"
        )

## 9. Understanding the DD Pass in the Fidelity-Optimal Strategy

When you use `QBCompiler.compile()` with the `fidelity_optimal` strategy, the compilation pipeline already includes calibration-aware mapping and gate optimization. The `compile_enhanced()` method adds DD as a post-processing step on top of Qiskit's transpilation.

The key insight: Qiskit does not enable DD at any optimization level. This is a free fidelity improvement that qb-compiler provides.

In [ ]:
from qb_compiler import QBCircuit

# Compare qb-compiler's standard compile() with compile_enhanced()
circ_qb = QBCircuit(5).h(0).cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4).measure_all()

# Standard compile (gate optimization only, no DD)
result_std = compiler.compile(circ_qb)

print(f"Standard compile():")
print(f"  Compiled depth:     {result_std.compiled_depth}")
print(f"  Estimated fidelity: {result_std.estimated_fidelity:.4f}")
print(f"  Pass log:")
for p in result_std.pass_log:
    print(f"    {p.pass_name:25s}  gates {p.gate_count_before}->{p.gate_count_after}")

In [ ]:
if target is not None:
    # Enhanced compile (Qiskit transpile + DD)
    ghz5_qiskit = QuantumCircuit(5, 5, name="GHZ-5")
    ghz5_qiskit.h(0)
    for i in range(4):
        ghz5_qiskit.cx(i, i + 1)
    ghz5_qiskit.measure(range(5), range(5))

    result_enh = compiler.compile_enhanced(ghz5_qiskit, target, n_seeds=10, dd_type="XX")

    print(f"\ncompile_enhanced():")
    print(f"  2Q gates (base):    {result_enh.two_qubit_gate_count}")
    print(f"  DD gates inserted:  {result_enh.dd_gates_inserted}")
    print(f"  Fidelity (base):    {result_enh.estimated_fidelity:.4f}")
    print(f"  Fidelity (DD):      {result_enh.estimated_fidelity_with_dd:.4f}")
    print(f"  Physical qubits:    {result_enh.physical_qubits}")
    if result_enh.calibration_timestamp:
        print(f"  Calibration date:   {result_enh.calibration_timestamp}")

## 10. Using the Low-Level DD API Directly

For advanced use cases, you can call `insert_dd()` directly on an already-transpiled Qiskit circuit.

In [ ]:
from qiskit import transpile

from qb_compiler.passes.scheduling.dynamical_decoupling import (
    insert_dd,
    insert_dd_calibration_aware,
)

if target is not None:
    # Step 1: Transpile with Qiskit
    ghz_circ = QuantumCircuit(4, 4, name="GHZ-4")
    ghz_circ.h(0)
    for i in range(3):
        ghz_circ.cx(i, i + 1)
    ghz_circ.measure(range(4), range(4))

    routed = transpile(ghz_circ, target=target, optimization_level=3, seed_transpiler=0)

    print(f"Routed circuit:")
    print(f"  Depth: {routed.depth()}")
    print(f"  Ops:   {routed.count_ops()}")

    # Step 2: Insert DD
    with_dd = insert_dd(routed, target, dd_type="XX")

    print(f"\nWith DD:")
    print(f"  Depth: {with_dd.depth()}")
    print(f"  Ops:   {with_dd.count_ops()}")

    # Count DD-added gates
    orig_ops = routed.count_ops()
    dd_ops = with_dd.count_ops()
    dd_x = dd_ops.get("x", 0) - orig_ops.get("x", 0)
    dd_y = dd_ops.get("y", 0) - orig_ops.get("y", 0)
    print(f"  DD X gates added: {dd_x}")
    print(f"  DD Y gates added: {dd_y}")

## 11. Calibration-Aware DD Selection

The `insert_dd_calibration_aware()` function uses per-qubit T2 data to automatically choose the best DD sequence. Qubits with short T2 (high decoherence risk) get XY4 for stronger suppression.

In [ ]:
if target is not None and compiler.calibration_properties is not None:
    routed = transpile(ghz_circ, target=target, optimization_level=3, seed_transpiler=0)

    with_dd_cal = insert_dd_calibration_aware(
        routed,
        target,
        compiler.calibration_properties,
        t2_threshold_us=50.0,
    )

    cal_ops = with_dd_cal.count_ops()
    orig_ops = routed.count_ops()
    dd_x_cal = cal_ops.get("x", 0) - orig_ops.get("x", 0)
    dd_y_cal = cal_ops.get("y", 0) - orig_ops.get("y", 0)

    sequence_used = "XY4" if dd_y_cal > 0 else "XX"
    print(f"Calibration-aware DD selected: {sequence_used}")
    print(f"  DD X gates: {dd_x_cal}")
    print(f"  DD Y gates: {dd_y_cal}")
    print(f"  Depth:      {with_dd_cal.depth()}")
else:
    print("Calibration data not available for this demo.")

## 12. Depth Impact of DD

DD fills idle slots with pulses, which does not typically increase the circuit's critical-path depth (it fills *existing* dead time). However, the scheduling pass may add delays that show up as increased depth in the Qiskit representation.

In [ ]:
if target is not None:
    print(f"{'Circuit':15s}  {'Depth(base)':>12s}  {'Depth(DD)':>12s}  {'Delta':>8s}")
    print("-" * 52)

    for n in [3, 5, 8, 10]:
        qc = QuantumCircuit(n, n, name=f"GHZ-{n}")
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
        qc.measure(range(n), range(n))

        routed = transpile(qc, target=target, optimization_level=3, seed_transpiler=0)
        with_dd = insert_dd(routed, target, dd_type="XX")

        d_base = routed.depth()
        d_dd = with_dd.depth()
        print(f"{qc.name:15s}  {d_base:>12d}  {d_dd:>12d}  {d_dd - d_base:>+8d}")

## 13. DD Decision Guide

| Circuit Type | DD Impact | Recommendation |
|-------------|-----------|----------------|
| **GHZ / linear chain** | Strong positive | Always use DD |
| **QFT** | Moderate positive | Use DD, XY4 for large circuits |
| **QAOA (ring)** | Moderate positive | Use DD |
| **VQE (brickwork)** | Small positive to neutral | Use DD, but don't expect large gains |
| **Dense parallel circuits** | Neutral to slight negative | Skip DD |
| **Single-qubit circuits** | None | DD not applicable (no idle periods) |

**Rule of thumb:** If your circuit has qubits that sit idle for more than a few gate durations, DD will help. The more asymmetric the circuit (some qubits active while others wait), the more DD helps.

## Summary

In this notebook you learned:

- **Dynamical Decoupling** — pulses inserted during idle periods to suppress decoherence
- **`compile_enhanced()`** — Qiskit transpilation + DD in one call, returns `EnhancedCompileResult`
- **XX vs XY4** — XX is lighter (2 pulses), XY4 is stronger (4 pulses, better for short-T2 qubits)
- **When DD helps** — circuits with long idle periods (GHZ chains, QFT, sequential algorithms)
- **When DD is neutral** — dense circuits with fully parallel gate schedules
- **`insert_dd()`** — low-level API for adding DD to already-routed circuits
- **`insert_dd_calibration_aware()`** — auto-selects DD type based on per-qubit T2 data
- **Depth impact** — DD fills existing idle time, does not extend the critical path
- **Key fact** — Qiskit does not enable DD by default; qb-compiler provides this as a guaranteed improvement